# Week 9: Multi-Agent

```mermaid
graph TD
    U[User Query] --> P[Supervisor · planner]
    P -->|"next=rag"| RAG[RAG Agent]
    P -->|"next=writing"| WRT[Writing Agent]

    RAG --> RP[rag_planner]
    RP -->|use_decomp=true| DEC[decompose]
    RP -->|use_decomp=false| RET[retrieve]
    DEC --> RET
    RET --> GRD[grade]
    GRD -->|yes| GEN[generate]
    GRD -->|"no + retry"| RWR[rewrite] --> RET
    GRD -->|"no + max"| GEN

    WRT --> E["안녕하세요 (placeholder)"]

    RAG --> OUT[output]
    WRT --> OUT
```


In [20]:
import os, sys, re, time, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

from typing import List, Any
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, END

from utils.doc_preprocessing import extract_sections, get_breadcrumb
from prompt.prompt import GRADE_PROMPT, REWRITE_PROMPT, GENERATE_PROMPT

load_dotenv("../.env")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [21]:
# pymupdf4llm 활용해 만든 VectorDB 호출

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

BASE_DIR = "C:/Users/seohyun/OneDrive/2026/Advanced_RAG"
PDF_PATH = os.path.join(BASE_DIR, 'data', 'registration_of_real_estatee_manual.pdf')
DENSE_DB_PATH = os.path.join(BASE_DIR, 'chroma_db', 'real_estatee_manual')
COLLECTION_NAME = 'real_estatee_manual'

embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

db = Chroma(
    persist_directory=DENSE_DB_PATH,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
)
print(f'기존 ChromaDB 로드: {db._collection.count()}개 문서')

기존 ChromaDB 로드: 327개 문서


In [22]:
def korean_tokenizer(text: str):
    """BM25용 한국어 토크나이저: 특수문자 제거 + 공백 분리 + 1글자 제거"""
    cleaned = re.sub("[^가-힣a-zA-Z0-9]", " ", text)
    return [t for t in cleaned.split() if len(t) > 1]
# BM25용 Document 리스트 생성
raw = db.get(include=["documents", "metadatas"])

bm25_docs = [
    Document(page_content=doc, metadata=metadata or {})
    for doc, metadata in zip(raw["documents"], raw["metadatas"])
]

bm25_retriever = BM25Retriever.from_documents(
    bm25_docs, 
    k=10, 
    preprocess_func=korean_tokenizer
    )

dense_retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10, "fetch_k": 20},
)

# Hybrid (reranker 없음)
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5], c=60,
)

def hybrid_search(query: str, top_k: int = 5) -> List[Document]:
    return hybrid_retriever.invoke(query)[:top_k]

### RAG Agent (with Sub-Query Decomposition)
planner가 `use_decomp=True`로 설정하면 `rag_planner → decompose → retrieve` 경로를 타고,
아니면 `rag_planner → retrieve`로 바로 진입한다.

In [23]:
class RagState(TypedDict):
    question: str
    rewritten_question: str
    sub_queries: List[str]  
    documents: List[Any]
    answer: str
    grade_result: str
    retry_count: int
    use_decomp: bool         

MAX_RETRIES = 2

class GradeResult(BaseModel):
    relevance: str = Field(description="'yes' 또는 'no'")
    reason: str = Field(description="판단 이유")

class SubQueries(BaseModel):
    queries: List[str] = Field(description="분해된 하위 질문 목록 (2~3개)")

grade_llm = llm.with_structured_output(GradeResult)
decomp_llm = llm.with_structured_output(SubQueries)

DECOMP_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """복잡한 질문을 독립적으로 검색 가능한 2~3개의 구체적인 하위 질문으로 분해하세요.
각 하위 질문은 등기 업무 키워드를 포함해 단독 검색이 가능해야 합니다.
재작성된 하위 질문 목록만 출력하세요."""),
    ("human", "질문: {question}"),
])


def rag_planner_node(state: RagState) -> dict:
    return {}  # entry hub — 라우팅만 담당

def route_from_rag_planner(state: RagState) -> str:
    return "decompose" if state.get("use_decomp") else "retrieve"

def decompose_node(state: RagState) -> dict:
    result = decomp_llm.invoke(DECOMP_PROMPT.format_messages(question=state["question"]))
    print(f"  [decompose] {result.queries}")
    return {"sub_queries": result.queries}

def retrieve(state: RagState) -> dict:
    sub_qs = state.get("sub_queries") or []
    if sub_qs:
        all_docs, seen, docs = [], set(), []
        for sq in sub_qs:
            all_docs.extend(hybrid_search(sq, top_k=3))
        for d in all_docs:
            if d.page_content not in seen:
                seen.add(d.page_content)
                docs.append(d)
        docs = docs[:5]
    else:
        q = state.get("rewritten_question") or state["question"]
        docs = hybrid_search(q, top_k=5)
    return {"documents": docs, "grade_result": "", "answer": ""}

def grade_documents(state: RagState) -> dict:
    q = state.get("rewritten_question") or state["question"]
    docs = state["documents"]
    if not docs:
        return {"grade_result": "no"}
    doc_previews = "\n\n".join(
        f"[문서 {i+1}] 출처:{d.metadata.get('breadcrumb', 'N/A')}\n{d.page_content[:250]}"
        for i, d in enumerate(docs[:5])
    )
    result = grade_llm.invoke(GRADE_PROMPT.format_messages(question=q, doc_previews=doc_previews))
    return {"grade_result": result.relevance}

def rewrite_query(state: RagState) -> dict:
    cur = state.get("rewritten_question") or state["question"]
    n = (state.get("retry_count") or 0) + 1
    rewritten = llm.invoke(REWRITE_PROMPT.format_messages(question=cur)).content.strip()
    # 재시도 시 분해 비활성화 — rewrite된 단일 쿼리로 재검색
    return {"rewritten_question": rewritten, "retry_count": n, "sub_queries": [], "use_decomp": False}

def generate(state: RagState) -> dict:
    q = state.get("rewritten_question") or state["question"]
    docs = state.get("documents") or []
    if state.get("grade_result") != "yes" or not docs:
        return {"answer": "(관련 문서 부족) 제공된 문서에서 확인할 수 없습니다."}
    context = "\n\n".join(d.page_content for d in docs)
    return {"answer": llm.invoke(GENERATE_PROMPT.format_messages(context=context, question=q)).content}

def route_after_grade(state: RagState) -> str:
    if state.get("grade_result") == "yes":
        return "generate"
    if (state.get("retry_count") or 0) >= MAX_RETRIES:
        return "generate"
    return "rewrite_query"

rag_wf = StateGraph(RagState)
rag_wf.add_node("rag_planner", rag_planner_node)
rag_wf.add_node("decompose", decompose_node)
rag_wf.add_node("retrieve", retrieve)
rag_wf.add_node("grade_documents", grade_documents)
rag_wf.add_node("rewrite_query", rewrite_query)
rag_wf.add_node("generate", generate)
rag_wf.set_entry_point("rag_planner")
rag_wf.add_conditional_edges("rag_planner", route_from_rag_planner,
                              {"decompose": "decompose", "retrieve": "retrieve"})
rag_wf.add_edge("decompose", "retrieve")
rag_wf.add_edge("retrieve", "grade_documents")
rag_wf.add_conditional_edges(
    "grade_documents", route_after_grade,
    {"generate": "generate", "rewrite_query": "rewrite_query"},
)
rag_wf.add_edge("rewrite_query", "retrieve")
rag_wf.add_edge("generate", END)
rag_app = rag_wf.compile()


def run_rag_agent(query: str, verbose: bool = False, use_decomp: bool = False) -> dict:
    final = rag_app.invoke({
        "question": query, "rewritten_question": "", "sub_queries": [],
        "documents": [], "answer": "", "grade_result": "", "retry_count": 0,
        "use_decomp": use_decomp,
    })
    docs = final.get("documents") or []
    if verbose:
        print(f"  [rag] grade={final.get('grade_result')} retry={final.get('retry_count')}")
    return {
        "answer": final["answer"],
        "contexts": [d.page_content for d in docs],
        "grade_result": final.get("grade_result", ""),
        "retry_count": final.get("retry_count", 0),
    }


### Writing Agent

In [24]:
# ── Writing Agent (placeholder) ───────────────────────────
def run_writing_agent(user_input: str) -> str:
    # TODO: 실제 이메일 초안 생성 (사용자가 입력한 수신자·서류 정보 활용)
    return "안녕하세요"

## 2. Supervisor

planner 가 user query 를 보고 `rag` / `writing` 중 하나를 결정한다.
결정된 에이전트로 라우팅 → 실행 → `output` 저장 → END.
cursor 없음. plan 리스트 없음.


In [25]:
# ── Plan ─────────────────────────────────────────────────
class Plan(BaseModel):
    next: str = Field(description="라우팅할 에이전트. 'rag' 또는 'writing'")
    use_decomp: bool = Field(default=False, description="복잡한 질문 시 RAG sub-query 분해")
    reason: str = Field(description="라우팅 이유")

planner_llm = llm.with_structured_output(Plan)

PLANNER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """당신은 셀프 등기 어시스턴트의 라우터입니다.

에이전트:
- rag     : 등기 절차·개념·서류 내용 질의응답 (내부 문서 검색 기반)
- writing : 건설사·은행 등에 보내는 이메일 작성

라우팅 기준:
- 정보/절차/개념/비교 질문  → rag
- 복잡하거나 다단계 질문    → rag,  use_decomp=true
- 이메일 작성 요청          → writing

이유를 한 줄로 설명하고, 계획을 구조화해 출력하세요."""),
    ("human", "사용자 요청: {user_input}"),
])

# ── State ─────────────────────────────────────────────────
class SupervisorState(TypedDict):
    user_input : str
    next_agent : str    # planner 가 결정: "rag" or "writing"
    use_decomp : bool
    output     : str    # 선택된 에이전트의 최종 출력
    route_history: List[str]


# ── Nodes ─────────────────────────────────────────────────
def planner_node(state: SupervisorState) -> dict:
    p = planner_llm.invoke(PLANNER_PROMPT.format_messages(user_input=state["user_input"]))
    agent = p.next if p.next in {"rag", "writing"} else "rag"
    print(f"[planner] → {agent}  use_decomp={p.use_decomp}")
    print(f"           reason: {p.reason}")
    return {"next_agent": agent, "use_decomp": p.use_decomp,
            "route_history": [f"planner→{agent}"]}

def rag_node(state: SupervisorState) -> dict:
    res = run_rag_agent(state["user_input"], use_decomp=state.get("use_decomp", False))
    return {"output": res["answer"],
            "route_history": state["route_history"] + ["rag"]}

def writing_node(state: SupervisorState) -> dict:
    result = run_writing_agent(state["user_input"])
    return {"output": result,
            "route_history": state["route_history"] + ["writing"]}

# ── Routing ───────────────────────────────────────────────
def route(state: SupervisorState) -> str:
    return state["next_agent"]   # "rag" 또는 "writing"

# ── Graph ─────────────────────────────────────────────────
sup = StateGraph(SupervisorState)
sup.add_node("planner", planner_node)
sup.add_node("rag",     rag_node)
sup.add_node("writing", writing_node)
sup.set_entry_point("planner")
sup.add_conditional_edges("planner", route, {"rag": "rag", "writing": "writing"})
sup.add_edge("rag",     END)
sup.add_edge("writing", END)
supervisor_app = sup.compile()


def run_assistant(user_input: str) -> dict:
    init: SupervisorState = {
        "user_input": user_input, "next_agent": "",
        "use_decomp": False, "output": "", "route_history": [],
    }
    final = supervisor_app.invoke(init)
    print("=" * 60)
    print(f"[요청] {user_input}")
    print(f"[경로] {' → '.join(final['route_history'])}")
    print("=" * 60)
    print(final["output"])
    return final

print("Supervisor 준비 완료")


Supervisor 준비 완료


In [26]:
# Supervisor 그래프 시각화
print(supervisor_app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	rag(rag)
	writing(writing)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	planner -.-> rag;
	planner -.-> writing;
	rag --> __end__;
	writing --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 3. 데모
- 시나리오 1: 등기 절차 질문   → rag
- 시나리오 2: 이메일 작성 요청 → writing


In [27]:
# 시나리오 1: 등기 절차 질문 → rag
_ = run_assistant("소유권이전등기 신청할 때 필요한 절차가 어떻게 되나요?")

[planner] → rag  use_decomp=False
           reason: 소유권이전등기 신청 절차에 대한 정보 요청이므로 RAG로 라우팅합니다.
[요청] 소유권이전등기 신청할 때 필요한 절차가 어떻게 되나요?
[경로] planner→rag → rag
소유권 이전 등기를 신청할 때 필요한 절차는 다음과 같습니다:

1. **신청인 및 등기 의무자 확인**: 소유권 이전 등기는 매매, 증여 등 다양한 원인에 따라 신청할 수 있으며, 각 경우에 따라 신청인(등기권리자)과 등기 의무자(등기할 자)의 관계가 다릅니다. 예를 들어, 증여에 의한 소유권 이전 등기는 증여자(등기 의무자)와 수증자(등기 권리자)가 공동으로 신청해야 합니다.

2. **신청서 작성 및 제출**: 신청인 또는 그 대리인이 등기소에 출석하여 신청 정보를 포함한 서면을 제출해야 합니다. 대리인이 변호사나 법무사인 경우, 해당 사무원이 등기소에 출석하여 서면을 제출할 수 있습니다(「부동산등기법」 제24조 제1항).

3. **제출 서류 준비**: 소유권 이전 등기를 신청할 때 필요한 서류는 다음과 같습니다:
   - 소유권을 증명하는 서면 (예: 토지대장 등본, 건축물대장 등본)
   - 신청인의 주소를 증명하는 서면 (예: 주민등록등본)
   - 법인의 경우 법인등기사항증명서 또는 부동산등기용 등록번호증명서
   - 주민등록번호가 없는 경우 부동산등기용 등록번호증명서

4. **신청 기간 준수**: 소유권 이전 등기는 계약 체결일로부터 60일 이내에 신청해야 하며, 계약이 취소되거나 해제된 경우에는 이 규정이 적용되지 않습니다(「부동산등기특별조치법」 제2조 제1항).

5. **국민주택채권 매입**: 등기를 신청하는 자는 국민주택채권을 매입해야 하며, 공유물을 공유지분율에 따라 분할해 이전하는 경우에는 매입할 필요가 없습니다(「주택도시기금법」 제8조 제1항).

이러한 절차를 통해 소유권 이전 등기를 신청할 수 있습니다.


In [ ]:
# 시나리오 2: 이메일 작성 요청 → writing
_ = run_assistant("건설사에 소유권이전등기에 필요한 서류를 요청하는 이메일 써줘")

In [ ]:
# Re-ranker 제거 전 ablation 스터디

import json, time, os
import pandas as pd

with open("../data/eval/golden_set_v1.json", encoding="utf-8") as f:
    golden_set = json.load(f)

golden_df = pd.DataFrame(golden_set)
print(f"Golden Set: {len(golden_df)}문항")
print(golden_df["q_type"].value_counts().to_string())
print("-" * 60)

rag_all = []
for _, row in golden_df.iterrows():
    q = row["question"]
    t0 = time.time()
    res = run_rag_agent(q)         
    lat = time.time() - t0
    rag_all.append({
        "id": int(row["id"]), "q_type": row["q_type"], "question": q,
        "answer": res["answer"], "contexts": res["contexts"],
        "grade_result": res["grade_result"], "retry_count": res["retry_count"],
        "latency": lat,
    })
    print(f'Q{int(row["id"]):02d} [{row["q_type"]:12s}] retry={res["retry_count"]} '
          f'grade={res["grade_result"]:3s} {lat:5.1f}s')

os.makedirs("../data/result", exist_ok=True)
with open("../data/result/0618_advanced_agentic_rag.json", "w", encoding="utf-8") as f:
    json.dump(rag_all, f, ensure_ascii=False, indent=2)

print(f"\n평균 latency: {sum(r['latency'] for r in rag_all) / len(rag_all):.1f}s")

## 5. 한계 & 다음 단계
- **가드레일 미적용**: prompt injection / jailbreak 방지 장치 없음 → LlamaFirewall PromptGuard 추가 예정.
- **planner 라우팅 정확도** 미평가: "rag" / "writing" 판단이 잘못되면 전체가 어긋남 → routing accuracy 지표 필요.
- **writing_agent 미구현**: 현재 "안녕하세요" placeholder → 수신자·필요서류 기반 이메일 초안 생성으로 교체 필요.
- **use_decomp 자동 판단 정확도**: 단순 질문에도 decomp가 켜지면 latency 상승 → fine-grained 기준 정의 필요.
